**Instructions**

You are provided with a data set titled ‘Lung Cancer Patient Health and Treatment Records’, which captures detailed information about patient demographics, diagnosis stage, lifestyle risk factors, comorbidities and treatment outcomes.



*Note: This data set was inspired by various healthcare-related learning resources and is not intended to represent real patient data. *

In [2]:
# Import warnings
import warnings
warnings.filterwarnings("ignore")

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [4]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("LungCancerPatientRecords") \
    .getOrCreate()

In [5]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [6]:


# Load the patient and treatment records
df_patients = spark.read.csv("/content/drive/MyDrive/UpgradDatafiles/Lung Cancer.csv", header=True, inferSchema=True)
display(df_patients.head())

Row(id=1, age=64.0, gender='Male', country='Sweden', diagnosis_date=datetime.date(2016, 4, 5), cancer_stage='Stage I', family_history='Yes', smoking_status='Passive Smoker', bmi=29.4, cholesterol_level=199, hypertension=0, asthma=0, cirrhosis=1, other_cancer=0, treatment_type='Chemotherapy', end_treatment_date=datetime.date(2017, 9, 10), survived=0)

In [7]:
df_patients.printSchema()
df_patients.show(5)

root
 |-- id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: string (nullable = true)
 |-- smoking_status: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- cholesterol_level: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- asthma: integer (nullable = true)
 |-- cirrhosis: integer (nullable = true)
 |-- other_cancer: integer (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: date (nullable = true)
 |-- survived: integer (nullable = true)

+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| id| age|gender|    country|diagnosis_date|cancer_stage|family_history|smoking

### **Task 1:** Write a function that removes duplicate rows, ensures correct data types for numerical and date columns and converts all ‘yes’/ ‘no’ type fields into 1/0 format.  

In [8]:

def clean_patient_data(df):

    # 1. Remove duplicate rows
    df_cleaned = df.dropDuplicates()

    # 2. Convert 'yes'/'no' type fields into 1/0 format
    # Based on schema and sample data, 'family_history' is a 'Yes'/'No' column.
    yes_no_cols = ['family_history']

    for column_name in yes_no_cols:
        if column_name in df_cleaned.columns:
            df_cleaned = df_cleaned.withColumn(
                column_name,
                when(col(column_name) == "Yes", 1)
                .when(col(column_name) == "No", 0)
                .otherwise(col(column_name).cast("integer")) # Cast to integer to handle potential nulls or mixed types if any
            )

    return df_cleaned

## Apply the cleaning function and display results

In [9]:
# Apply the cleaning function
df_patients_cleaned = clean_patient_data(df_patients)

# Display the schema and some rows of the cleaned DataFrame to verify
print("Schema of cleaned DataFrame:")
df_patients_cleaned.printSchema()

print("\nFirst 5 rows of cleaned DataFrame:")
df_patients_cleaned.show(5)

print(f"\nOriginal number of rows: {df_patients.count()}")
print(f"Number of rows after cleaning: {df_patients_cleaned.count()}")

Schema of cleaned DataFrame:
root
 |-- id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: integer (nullable = true)
 |-- smoking_status: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- cholesterol_level: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- asthma: integer (nullable = true)
 |-- cirrhosis: integer (nullable = true)
 |-- other_cancer: integer (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: date (nullable = true)
 |-- survived: integer (nullable = true)


First 5 rows of cleaned DataFrame:
+---+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| id| age|

## Missing Value Analysis

In [10]:


# Check for missing values in each column
def check_missing_values(df):
    print("\nMissing values count per column:")
    df.select([
        sum(when(col(c).isNull(), 1)).alias(c)
        for c in df.columns
    ]).show()

check_missing_values(df_patients_cleaned)


Missing values count per column:
+----+----+------+-------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|  id| age|gender|country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+----+----+------+-------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|NULL|NULL|  NULL|   NULL|          NULL|        NULL|          NULL|          NULL|NULL|             NULL|        NULL|  NULL|     NULL|        NULL|          NULL|              NULL|    NULL|
+----+----+------+-------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+------

### **Task 2:** Write a function that adds a new column treatment_duration_days, which calculates the number of days between the diagnosis and the end of treatment. Then, return the average treatment duration for each treatment type.


### Calculate Treatment Duration and Average Duration per Treatment Type

In [11]:
def analyze_treatment_duration(df):

    # Add treatment_duration_days column
    df_with_duration = df.withColumn(
        "treatment_duration_days",
        datediff(col("end_treatment_date"), col("diagnosis_date"))
    )

    # Calculate average treatment duration for each treatment type
    avg_duration_by_type = df_with_duration.groupBy("treatment_type") \
                                           .agg(avg("treatment_duration_days").alias("average_treatment_duration_days")) \
                                           .orderBy("treatment_type")

    return avg_duration_by_type

### Apply the function and display results

In [12]:
# Apply the function to the cleaned DataFrame
avg_treatment_durations = analyze_treatment_duration(df_patients_cleaned)

# Display the results
avg_treatment_durations.show()

+--------------+-------------------------------+
|treatment_type|average_treatment_duration_days|
+--------------+-------------------------------+
|  Chemotherapy|             458.39540091909953|
|      Combined|              457.8152186120058|
|     Radiation|             458.40320462900917|
|       Surgery|             457.73744630723684|
+--------------+-------------------------------+



### **Task 3:** Write a function that returns the smoking_status group with the highest survival rate.  

### Smoking Status with Highest Survival Rate

In [13]:


def get_highest_survival_smoking_status(df):

    # Calculate survival rate for each smoking status
    survival_rates = df.groupBy("smoking_status") \
                       .agg(avg(col("survived")).alias("survival_rate"))

    # Find the smoking status with the highest survival rate
    highest_survival_status = survival_rates.orderBy(col("survival_rate").desc()).first()

    if highest_survival_status:
        return highest_survival_status["smoking_status"]
    else:
        return None

### Apply the function and display results

In [14]:
# Apply the function to the cleaned DataFrame
highest_survival_group = get_highest_survival_smoking_status(df_patients_cleaned)

# Display the result
if highest_survival_group:
    print(f"The smoking status group with the highest survival rate is: {highest_survival_group}")
else:
    print("Could not determine the smoking status group with the highest survival rate.")

The smoking status group with the highest survival rate is: Never Smoked


##**Task 4:** Write a function that returns the top three countries with the highest percentage of patients diagnosed in Stage IV.  

In [15]:

def get_top_countries_stage_iv(df):

    # Calculate total patients per country
    total_patients_per_country = df.groupBy("country").agg(count("id").alias("total_patients"))

    # Calculate Stage IV patients per country
    stage_iv_patients_per_country = df.filter(col("cancer_stage") == "Stage IV") \
                                      .groupBy("country").agg(count("id").alias("stage_iv_patients"))

    # Join and calculate percentage
    country_stage_iv_percentage = total_patients_per_country.join(
        stage_iv_patients_per_country,
        "country",
        "left_outer"
    ).withColumn(
        "stage_iv_percentage",
        round((col("stage_iv_patients") / col("total_patients")) * 100, 2)
    ).orderBy(col("stage_iv_percentage").desc())

    # Handle countries with no Stage IV patients (i.e., null percentage)
    country_stage_iv_percentage = country_stage_iv_percentage.na.fill(0, subset=["stage_iv_percentage"])

    # Get the top 3 countries
    top_3_countries = [row.country for row in country_stage_iv_percentage.limit(3).collect()]

    return top_3_countries

### Apply the function and display results

In [16]:
# Apply the function to the cleaned DataFrame
top_countries = get_top_countries_stage_iv(df_patients_cleaned)

# Display the result
if top_countries:
    print(f"The top three countries with the highest percentage of Stage IV diagnoses are: {', '.join(top_countries)}")
else:
    print("Could not determine the top countries with Stage IV diagnoses.")

The top three countries with the highest percentage of Stage IV diagnoses are: Greece, Croatia, Czech Republic


### **Task 5:** Write a function that filters patients who:  


*   Are male
*   Diagnosed in Stage III or IV
*   Have a family history of cancer  
*   Are current smokers
*   Have a BMI > 30
*   Survived
*   Return the average age and the percentage of these patients who had hypertension.

In [17]:
def analyze_filtered_patients(df):

    # Filter patients based on criteria
    filtered_df = df.filter(
        (col("gender") == "Male") &
        ((col("cancer_stage") == "Stage III") | (col("cancer_stage") == "Stage IV")) &
        (col("family_history") == 1) &
        (col("smoking_status") == "Current Smoker") &
        (col("bmi") > 30) &
        (col("survived") == 1)
    )

    num_filtered_patients = filtered_df.count()

    if num_filtered_patients == 0:
        print("No patients match the specified criteria.")
        return None, None

    # Calculate average age
    average_age = filtered_df.agg(avg(col("age"))).first()[0]

    # Calculate percentage of patients with hypertension
    hypertension_count = filtered_df.filter(col("hypertension") == 1).count()
    percentage_hypertension = (hypertension_count / num_filtered_patients) * 100

    return average_age, percentage_hypertension

### Apply the function and display results

In [18]:
# Apply the function to the cleaned DataFrame
average_age_result, percentage_hypertension_result = analyze_filtered_patients(df_patients_cleaned)

# Display the results
if average_age_result is not None:
    print(f"Average age of filtered patients: {average_age_result:.2f} years")
    print(f"Percentage of filtered patients with hypertension: {percentage_hypertension_result:.2f}%")

Average age of filtered patients: 55.18 years
Percentage of filtered patients with hypertension: 74.77%
